In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import harmonypy
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

ad.settings.zarr_write_format = 2

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "00_preprocess_and_cluster"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW = BASE.parent / "xenium_data_human" / "integrated" / "integrated_human.zarr" / "tables" / "table"

In [ ]:
adata = ad.read_zarr(RAW)
adata

In [ ]:
roi_name = {
    "C-ion_Late": "Cion_Late",
    "Xray_Late": "Xray_Late",
    "SF1": "SF1",
    "SF2": "SF2",
    "Xray_Under": "Xray_Under",
}
ORDER = ["C-ion_Late", "SF1", "SF2", "Xray_Late", "Xray_Under"]
order = pd.DataFrame({"roi": pd.Categorical(adata.obs["roi_id"].astype(str), ORDER, ordered=True),
                       "cell_id": adata.obs["cell_id"].astype(str)}, index=adata.obs_names)
adata = adata[order.sort_values(["roi", "cell_id"]).index].copy()
adata.obs["roi_id"] = adata.obs["roi_id"].astype(str)
adata.obs["roi_name"] = adata.obs["roi_id"].map(roi_name)
adata.obs["roi_name"].value_counts()

In [ ]:
CUT = {"n_genes_by_counts": (5, None), "total_counts": (10, None)}
MIN_CELLS = 3

sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
before = adata.obs[["roi_name", "total_counts", "n_genes_by_counts"]].copy()
before.to_parquet(OUT_DIR / "qc_per_cell_before.parquet")
n_genes_before = adata.n_vars
keep = np.ones(adata.n_obs, bool)
for col, (lo, hi) in CUT.items():
    if lo is not None:
        keep &= adata.obs[col].to_numpy() >= lo
    if hi is not None:
        keep &= adata.obs[col].to_numpy() <= hi
adata = adata[keep].copy()
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
adata.obs["roi_name"].value_counts()

In [ ]:
ROIS = ["SF1", "SF2", "Xray_Under", "Xray_Late", "Cion_Late"]
after = adata.obs
qc = pd.DataFrame({
    "cells_before": before["roi_name"].value_counts(),
    "cells_after": after["roi_name"].value_counts(),
    "median_transcripts_before": before.groupby("roi_name")["total_counts"].median(),
    "median_transcripts_after": after.groupby("roi_name")["total_counts"].median(),
    "median_genes_before": before.groupby("roi_name")["n_genes_by_counts"].median(),
    "median_genes_after": after.groupby("roi_name")["n_genes_by_counts"].median(),
}).reindex(ROIS)
qc.loc["All"] = [len(before), len(after), before["total_counts"].median(), after["total_counts"].median(),
                 before["n_genes_by_counts"].median(), after["n_genes_by_counts"].median()]
qc.insert(2, "removed_pct", ((1 - qc["cells_after"] / qc["cells_before"]) * 100).round(1))
qc[["cells_before", "cells_after"]] = qc[["cells_before", "cells_after"]].astype(int)
qc.to_csv(OUT_DIR / "qc_summary.csv")
total = pd.DataFrame({"before": [len(before), n_genes_before], "after": [adata.n_obs, adata.n_vars]},
                     index=["cells", "genes"])
total.to_csv(OUT_DIR / "qc_total.csv")
display(total)
qc

In [ ]:
def qc_violin(ax, values, labels, cutoff, ylabel, colors):
    data = [np.log10(np.asarray(v, dtype=float) + 1) for v in values]
    xs = np.arange(1, len(data) + 1)
    parts = ax.violinplot(data, positions=xs, widths=0.8, showextrema=False)
    for body, color in zip(parts["bodies"], colors):
        body.set(facecolor=color, edgecolor=fs.BLACK, linewidth=0.3, alpha=1)
    ax.hlines([np.median(d) for d in data], xs - 0.25, xs + 0.25, color=fs.BLACK, lw=0.75)
    for c in cutoff:
        if c is not None:
            ax.axhline(np.log10(c + 1), color=fs.GREY, lw=0.6, ls=(0, (3, 2)))
    top = max(d.max() for d in data)
    ticks = [t for t in [0, 1, 10, 100, 1000, 10000, 100000] if np.log10(t + 1) <= top * 1.02]
    ax.set_yticks(np.log10(np.array(ticks) + 1), [f"{t:,}" for t in ticks])
    ax.set_xticks(xs, labels)
    ax.set_xlim(0.4, len(data) + 0.6)
    ax.set_ylabel(ylabel)

METRIC = {"total_counts": "Transcripts per cell", "n_genes_by_counts": "Genes per cell"}

fig, axes = plt.subplots(2, 1, figsize=(fs.COL1, 60 * fs.MM), sharex=True, layout="constrained")
fig.get_layout_engine().set(h_pad=0.5 * fs.MM, hspace=0)
for ax, (col, ylabel) in zip(axes, METRIC.items()):
    qc_violin(ax, [before.loc[before["roi_name"] == r, col] for r in ROIS], [fs.label(r) for r in ROIS],
                 CUT[col], ylabel, colors=[fs.ROI_COLORS[r] for r in ROIS])
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
fig.align_ylabels(axes)
fs.save(fig, OUT_DIR / "qc_violin_by_roi")
plt.show()

In [ ]:
with threadpool_limits(limits=1):
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=100)
    sc.pp.log1p(adata)
    adata.layers["log1p_norm"] = adata.X.copy()
    adata.raw = adata
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=50, svd_solver="arpack", random_state=42)

In [ ]:
with threadpool_limits(limits=1):
    ho = harmonypy.run_harmony(np.ascontiguousarray(adata.obsm["X_pca"], dtype=np.float64), adata.obs[["roi_name"]].astype(str),
                               ["roi_name"], max_iter_harmony=20, random_state=42, device="cpu")
adata.obsm["X_pca_harmony"] = np.ascontiguousarray(ho.Z_corr, dtype=np.float32)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40, use_rep="X_pca_harmony", metric="cosine", random_state=42)
sc.tl.leiden(adata, resolution=2.1, flavor="leidenalg", random_state=42)
sc.tl.umap(adata, min_dist=0.3, spread=1.0, random_state=42)
ax = sc.pl.umap(adata, color="leiden", legend_loc="on data", legend_fontsize=6, legend_fontweight="normal",
                frameon=False, title="", size=0.5, show=False)
fig = ax.figure
fig.set_size_inches(fs.COL1, fs.COL1)
fs.save(fig, OUT_DIR / "umap_leiden")
plt.show()
adata.obs["leiden"].value_counts().sort_index()

In [ ]:
ax = sc.pl.umap(adata, color="roi_name", frameon=False, title="", size=0.5, show=False)
for t in ax.get_legend().get_texts():
    t.set_text(fs.label(t.get_text()))
fig = ax.figure
fig.set_size_inches(fs.COL1, fs.COL1)
fs.save(fig, OUT_DIR / "umap_roi")
plt.show()

In [ ]:
# adata.write_zarr(BASE / "data" / "integrated_human_preprocessed.zarr")